# 01 - Mine Dependabot Evidence

This notebook mines repo-level Dependabot evidence for the downstream repositories in the RQ2 pairwise dataset.

It intentionally produces raw evidence only. Classification by adoption date happens in `02_classify_dependabot_usage.ipynb`.

## Inputs and Outputs

Input:

- `../../data/rq2/rq2_master_pairwise.csv`

Outputs:

- `data/dependabot_repo_evidence_raw.csv`
- `data/dependabot_repo_mining_errors.csv`

GitHub authentication:

- Paste a token into the token cell below, or
- Use `GITHUB_TOKEN` from the environment, or
- Fallback to local `.token` in this notebook directory.

## GitHub Token

Paste your GitHub token between the quotes below before running the mining cells. Leave it blank if you want to use `GITHUB_TOKEN` or `.token` instead.

In [18]:
PASTED_GITHUB_TOKEN = ''

In [19]:
from pathlib import Path
import json
import os
import re
import time
import urllib.error
import urllib.parse
import urllib.request

import pandas as pd

ROOT = Path.cwd().parents[1]
INPUT_CSV = ROOT / 'data' / 'rq2' / 'rq2_master_pairwise.csv'
DATA_DIR = ROOT / 'data' / 'rq3'
RAW_EVIDENCE_CSV = DATA_DIR / 'dependabot_repo_evidence_raw.csv'
ERRORS_CSV = DATA_DIR / 'dependabot_repo_mining_errors.csv'

DATA_DIR.mkdir(parents=True, exist_ok=True)

TOKEN_FILE = ROOT / '.token'
if not TOKEN_FILE.exists() and (ROOT / 'dependabot' / '.token').exists():
    TOKEN_FILE = ROOT / 'dependabot' / '.token'

PASTED_TOKEN = globals().get('PASTED_GITHUB_TOKEN', '').strip()
GITHUB_TOKEN = None
TOKEN_SOURCE = None
if PASTED_TOKEN:
    GITHUB_TOKEN = PASTED_TOKEN
    TOKEN_SOURCE = 'PASTED_GITHUB_TOKEN cell'
elif os.environ.get('GITHUB_TOKEN'):
    GITHUB_TOKEN = os.environ['GITHUB_TOKEN'].strip()
    TOKEN_SOURCE = 'GITHUB_TOKEN environment variable'
elif TOKEN_FILE.exists():
    GITHUB_TOKEN = TOKEN_FILE.read_text().strip()
    TOKEN_SOURCE = str(TOKEN_FILE)

if not GITHUB_TOKEN:
    raise RuntimeError('Set PASTED_GITHUB_TOKEN, GITHUB_TOKEN, or create a .token file before running this notebook.')

print(f'Using GitHub token from: {TOKEN_SOURCE}')

REST_API = 'https://api.github.com'
GRAPHQL_API = 'https://api.github.com/graphql'

DEPENDABOT_AUTHORS = [
    'app/dependabot',
    'app/dependabot-preview',
    'dependabot[bot]',
    'dependabot-preview[bot]',
]

CONFIG_PATHS = [
    '.github/dependabot.yml',
    '.github/dependabot.yaml',
]

# Dependabot was introduced in 2017, so ignore impossible earlier evidence.
MIN_DEPENDABOT_EVIDENCE_DATE = '2017-01-01T00:00:00Z'
MIN_DEPENDABOT_SEARCH_DATE = '2017-01-01'

REQUEST_DELAY_SECONDS = 0.2
MAX_RETRIES = 4
REFRESH_EXISTING = False

Using GitHub token from: PASTED_GITHUB_TOKEN cell


## Load and Normalize Repositories

In [20]:
pairs = pd.read_csv(INPUT_CSV)
required_cols = {'CVE', 'upstream_GA', 'downstream_GA', 'downstream_repo', 'commit_url', 'adoption_date'}
missing = required_cols - set(pairs.columns)
if missing:
    raise ValueError(f'Missing required columns: {sorted(missing)}')

def normalize_repo(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    if not text:
        return None

    text = text.replace('git@github.com:', 'https://github.com/')
    text = text.replace('http://github.com/', 'https://github.com/')
    if text.startswith('https://github.com/'):
        text = text[len('https://github.com/'):]
    if text.startswith('github.com/'):
        text = text[len('github.com/'):]
    text = text.split('#', 1)[0].split('?', 1)[0].strip('/')
    if text.endswith('.git'):
        text = text[:-4]

    parts = [p for p in text.split('/') if p]
    if len(parts) < 2:
        return None
    return f'{parts[0]}/{parts[1]}'

pairs['normalized_downstream_repo'] = pairs['downstream_repo'].map(normalize_repo)
repos = sorted(pairs['normalized_downstream_repo'].dropna().unique())

print(f'Pairwise rows: {len(pairs):,}')
print(f'Unique downstream repositories: {len(repos):,}')
pairs[['downstream_repo', 'normalized_downstream_repo']].drop_duplicates().head()

Pairwise rows: 1,677
Unique downstream repositories: 742


,downstream_repo,normalized_downstream_repo
0,btheu/estivate,btheu/estivate
1,jcabi/jcabi-http,jcabi/jcabi-http
2,ashwanthkumar/gocd-java-client,ashwanthkumar/gocd-java-client
3,grasshopper7/pdfextentreporter,grasshopper7/pdfextentreporter
4,kuehne-trustable-de/ca3sCore,kuehne-trustable-de/ca3sCore


## GitHub API Helpers

In [21]:
def github_headers(extra=None):
    headers = {
        'Authorization': f'Bearer {GITHUB_TOKEN}',
        'Accept': 'application/vnd.github+json',
        'X-GitHub-Api-Version': '2022-11-28',
        'User-Agent': 'rq3-dependabot-miner',
    }
    if extra:
        headers.update(extra)
    return headers

def sleep_for_rate_limit(headers):
    remaining = headers.get('X-RateLimit-Remaining')
    reset = headers.get('X-RateLimit-Reset')
    if remaining == '0' and reset:
        wait = max(0, int(reset) - int(time.time())) + 5
        print(f'Rate limit reached. Sleeping {wait} seconds.')
        time.sleep(wait)

def request_json(url, data=None, method=None):
    body = None
    headers = github_headers()
    if data is not None:
        body = json.dumps(data).encode('utf-8')
        headers['Content-Type'] = 'application/json'

    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        req = urllib.request.Request(url, data=body, headers=headers, method=method)
        try:
            with urllib.request.urlopen(req, timeout=60) as response:
                payload = json.load(response)
                sleep_for_rate_limit(response.headers)
                time.sleep(REQUEST_DELAY_SECONDS)
                return payload, dict(response.headers)
        except urllib.error.HTTPError as exc:
            err_body = exc.read().decode('utf-8', errors='replace')
            last_error = {'status': exc.code, 'body': err_body[:1000], 'url': url}
            if exc.code in {403, 429, 500, 502, 503, 504}:
                sleep_for_rate_limit(exc.headers)
                time.sleep(min(60, 2 ** attempt))
                continue
            raise
        except Exception as exc:
            last_error = {'status': type(exc).__name__, 'body': str(exc), 'url': url}
            time.sleep(min(60, 2 ** attempt))

    raise RuntimeError(f'GitHub request failed after retries: {last_error}')

def graphql(query, variables=None):
    payload = {'query': query, 'variables': variables or {}}
    data, headers = request_json(GRAPHQL_API, data=payload, method='POST')
    if data.get('errors'):
        raise RuntimeError(json.dumps(data['errors'])[:2000])
    return data['data'], headers

## Validate GitHub Token

Run this before the mining loop. A `401 Unauthorized` here means the token is invalid, expired, revoked, copied incorrectly, or not accepted by GitHub for API access.

In [22]:
def validate_github_token():
    try:
        user, _ = request_json(f'{REST_API}/user')
        gql_data, _ = graphql('query { viewer { login } rateLimit { remaining resetAt } }')
    except urllib.error.HTTPError as exc:
        if exc.code == 401:
            raise RuntimeError(
                f'GitHub returned 401 Unauthorized for token source: {TOKEN_SOURCE}. '
                'Use a valid, unexpired GitHub token and rerun from the token/setup cells.'
            ) from exc
        raise

    print(f'Authenticated REST user: {user.get("login")}')
    print(f'Authenticated GraphQL user: {gql_data["viewer"]["login"]}')
    print(f'GraphQL remaining requests: {gql_data["rateLimit"]["remaining"]}')

validate_github_token()

Authenticated REST user: kaziamity
Authenticated GraphQL user: kaziamity
GraphQL remaining requests: 4999


## Evidence Mining Functions

In [23]:
FIRST_PR_QUERY = '''
query($q: String!) {
  search(query: $q, type: ISSUE, first: 1) {
    issueCount
    nodes {
      ... on PullRequest {
        createdAt
        number
        title
        url
        author { login }
      }
    }
  }
  rateLimit { cost remaining resetAt }
}
'''

def first_dependabot_pr(repo):
    candidates = []
    searched = []
    for author in DEPENDABOT_AUTHORS:
        search_query = f'repo:{repo} is:pr author:{author} created:>={MIN_DEPENDABOT_SEARCH_DATE} sort:created-asc'
        searched.append(search_query)
        data, _ = graphql(FIRST_PR_QUERY, {'q': search_query})
        nodes = data['search']['nodes']
        if nodes:
            pr = nodes[0]
            candidates.append({
                'createdAt': pr.get('createdAt'),
                'number': pr.get('number'),
                'title': pr.get('title'),
                'url': pr.get('url'),
                'author': (pr.get('author') or {}).get('login'),
                'matched_author_query': author,
                'issueCount': data['search'].get('issueCount'),
            })

    if not candidates:
        return {
            'has_dependabot_pr': False,
            'first_dependabot_pr_date': None,
            'first_dependabot_pr_url': None,
            'first_dependabot_pr_number': None,
            'first_dependabot_pr_author': None,
            'first_dependabot_pr_title': None,
            'first_dependabot_pr_query_author': None,
            'dependabot_pr_search_queries': ' | '.join(searched),
        }

    earliest = sorted(candidates, key=lambda item: item['createdAt'])[0]
    return {
        'has_dependabot_pr': True,
        'first_dependabot_pr_date': earliest['createdAt'],
        'first_dependabot_pr_url': earliest['url'],
        'first_dependabot_pr_number': earliest['number'],
        'first_dependabot_pr_author': earliest['author'],
        'first_dependabot_pr_title': earliest['title'],
        'first_dependabot_pr_query_author': earliest['matched_author_query'],
        'dependabot_pr_search_queries': ' | '.join(searched),
    }

def parse_last_link_page(link_header):
    if not link_header:
        return None
    match = re.search(r'<([^>]+)>;\s*rel="last"', link_header)
    if not match:
        return None
    return match.group(1)

def commits_for_path(repo, path, url=None):
    if url is None:
        encoded_path = urllib.parse.quote(path, safe='')
        since = urllib.parse.quote(MIN_DEPENDABOT_EVIDENCE_DATE, safe='')
        url = f'{REST_API}/repos/{repo}/commits?path={encoded_path}&since={since}&per_page=1'
    return request_json(url)

def first_config_commit_for_path(repo, path):
    data, headers = commits_for_path(repo, path)
    if not data:
        return None

    last_url = parse_last_link_page(headers.get('Link', ''))
    if last_url:
        last_page_data, _ = commits_for_path(repo, path, url=last_url)
        if not last_page_data:
            return None
        commit = last_page_data[-1]
    else:
        commit = data[-1]

    commit_info = commit.get('commit', {})
    return {
        'path': path,
        'sha': commit.get('sha'),
        'html_url': commit.get('html_url'),
        'author_date': (commit_info.get('author') or {}).get('date'),
        'committer_date': (commit_info.get('committer') or {}).get('date'),
        'message': commit_info.get('message'),
    }

def first_dependabot_config(repo):
    candidates = []
    for path in CONFIG_PATHS:
        item = first_config_commit_for_path(repo, path)
        if item:
            candidates.append(item)

    if not candidates:
        return {
            'has_dependabot_config': False,
            'dependabot_config_path': None,
            'dependabot_config_created_date': None,
            'dependabot_config_created_sha': None,
            'dependabot_config_created_url': None,
            'dependabot_config_created_message': None,
        }

    earliest = sorted(candidates, key=lambda item: item.get('committer_date') or item.get('author_date') or '')[0]
    return {
        'has_dependabot_config': True,
        'dependabot_config_path': earliest['path'],
        'dependabot_config_created_date': earliest.get('committer_date') or earliest.get('author_date'),
        'dependabot_config_created_sha': earliest['sha'],
        'dependabot_config_created_url': earliest['html_url'],
        'dependabot_config_created_message': earliest.get('message'),
    }

def mine_repo(repo):
    result = {
        'repo': repo,
        'repo_url': f'https://github.com/{repo}',
        'mined_at_utc': pd.Timestamp.utcnow().isoformat(),
        'mining_status': 'ok',
        'error_type': None,
        'error_message': None,
    }
    result.update(first_dependabot_pr(repo))
    result.update(first_dependabot_config(repo))
    return result

## Resume Cache

In [24]:
if RAW_EVIDENCE_CSV.exists():
    evidence = pd.read_csv(RAW_EVIDENCE_CSV)
else:
    evidence = pd.DataFrame()

if ERRORS_CSV.exists():
    errors = pd.read_csv(ERRORS_CSV)
else:
    errors = pd.DataFrame()

completed = set()
if not evidence.empty and not REFRESH_EXISTING:
    completed.update(evidence.loc[evidence['mining_status'].eq('ok'), 'repo'].dropna())

remaining = [repo for repo in repos if REFRESH_EXISTING or repo not in completed]

print(f'Already mined successfully: {len(completed):,}')
print(f'Remaining repositories: {len(remaining):,}')

Already mined successfully: 0
Remaining repositories: 742


## Run Mining

This cell appends after each repository so the notebook can be stopped and resumed.

In [25]:
def append_csv_row(path, row):
    row_df = pd.DataFrame([row])
    write_header = not path.exists()
    row_df.to_csv(path, mode='a', header=write_header, index=False)

for idx, repo in enumerate(remaining, start=1):
    print(f'[{idx:,}/{len(remaining):,}] {repo}')
    try:
        row = mine_repo(repo)
        append_csv_row(RAW_EVIDENCE_CSV, row)
    except Exception as exc:
        err = {
            'repo': repo,
            'repo_url': f'https://github.com/{repo}',
            'mined_at_utc': pd.Timestamp.utcnow().isoformat(),
            'mining_status': 'error',
            'error_type': type(exc).__name__,
            'error_message': str(exc)[:2000],
        }
        append_csv_row(ERRORS_CSV, err)
        print(f'  ERROR: {err["error_type"]}: {err["error_message"][:300]}')

[1/742] 42BV/jarb
[2/742] 52North/arctic-sea
[3/742] 52North/series-hibernate
[4/742] 6tail/nlf2-maven
[5/742] 88250/latke
[6/742] AKSW/jena-sparql-api
[7/742] Activiti/Activiti
[8/742] AlejandroRivera/embedded-rabbitmq
[9/742] AmyAssist/Amy
[10/742] AquaticInformatics/aquarius-sdk-java
[11/742] ArcBees/GWTP
[12/742] ArcadeData/arcade-connectors
[13/742] AsyncHttpClient/async-http-client
[14/742] AthenZ/athenz
[15/742] AxonFramework/AxonFramework
[16/742] Axway/ats-framework
[17/742] Azure/azure-sdk-for-java
[18/742] Baidu-ecom/Jprotobuf-rpc-socket
[19/742] BaseXdb/basex
[20/742] Bedework/bw-calendar-engine
[21/742] Bedework/bw-event-registration
[22/742] Bedework/bw-synch
[23/742] BroadleafCommerce/BroadleafCommerce
[24/742] BrunoEberhard/minimal-j
[25/742] Bynder/bynder-java-sdk
[26/742] CAFAudit/audit-service
[27/742] Captain-P-Goldfish/SCIM
[28/742] CloudSlang/cloud-slang
[29/742] CloudSlang/cs-actions
[30/742] CloudSlang/score
[31/742] Comcast/jrugged
[32/742] ConsumerDataStandard

## Quick Summary

In [26]:
evidence = pd.read_csv(RAW_EVIDENCE_CSV) if RAW_EVIDENCE_CSV.exists() else pd.DataFrame()
errors = pd.read_csv(ERRORS_CSV) if ERRORS_CSV.exists() else pd.DataFrame()

print(f'Evidence rows: {len(evidence):,}')
print(f'Error rows: {len(errors):,}')

if not evidence.empty:
    print('\nDependabot PR evidence:')
    print(evidence['has_dependabot_pr'].value_counts(dropna=False))
    print('\nDependabot config evidence:')
    print(evidence['has_dependabot_config'].value_counts(dropna=False))
    display(evidence.head())

if not errors.empty:
    print('\nTop error types:')
    print(errors['error_type'].value_counts(dropna=False).head(10))
    display(errors.head())

Evidence rows: 742
Error rows: 443

Dependabot PR evidence:
has_dependabot_pr
True     545
False    197
Name: count, dtype: int64

Dependabot config evidence:
has_dependabot_config
False    533
True     209
Name: count, dtype: int64


,repo,repo_url,mined_at_utc,mining_status,error_type,error_message,has_dependabot_pr,first_dependabot_pr_date,first_dependabot_pr_url,first_dependabot_pr_number,first_dependabot_pr_author,first_dependabot_pr_title,first_dependabot_pr_query_author,dependabot_pr_search_queries,has_dependabot_config,dependabot_config_path,dependabot_config_created_date,dependabot_config_created_sha,dependabot_config_created_url,dependabot_config_created_message
0,42BV/jarb,https://github.com/42BV/jarb,2026-08-14T17:00:36.520962+00:00,ok,NaN,NaN,True,2020-01-21T21:11:51Z,https://github.com/42BV/jarb/pull/53,53.0,dependabot,Bump spring.version from 5.1.3.RELEASE to 5.2....,app/dependabot,repo:42BV/jarb is:pr author:app/dependabot cre...,False,NaN,NaN,NaN,NaN,NaN
1,52North/arctic-sea,https://github.com/52North/arctic-sea,2026-08-14T17:00:42.593324+00:00,ok,NaN,NaN,True,2019-08-01T21:59:32Z,https://github.com/52North/arctic-sea/pull/47,47.0,dependabot,Bump version.jackson from 2.9.9 to 2.10.0.pr1,app/dependabot,repo:52North/arctic-sea is:pr author:app/depen...,True,.github/dependabot.yml,2021-04-29T15:25:46Z,fe7416a0a9a56c960616fe58de41f1e5b0e19049,https://github.com/52North/arctic-sea/commit/f...,Upgrade to GitHub-native Dependabot
2,52North/series-hibernate,https://github.com/52North/series-hibernate,2026-08-14T17:00:50.205286+00:00,ok,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,repo:52North/series-hibernate is:pr author:app...,True,.github/dependabot.yml,2021-04-29T15:25:57Z,924fab33f054d8521a2e2ade95609ec9f0aba51b,https://github.com/52North/sensorweb-server-db...,Upgrade to GitHub-native Dependabot
3,6tail/nlf2-maven,https://github.com/6tail/nlf2-maven,2026-08-14T17:00:59.268097+00:00,ok,NaN,NaN,True,2019-10-29T21:06:00Z,https://github.com/6tail/nlf2-maven/pull/1,1.0,dependabot,Bump c3p0 from 0.9.5.2 to 0.9.5.4 in /nlf2-plu...,app/dependabot,repo:6tail/nlf2-maven is:pr author:app/dependa...,False,NaN,NaN,NaN,NaN,NaN
4,88250/latke,https://github.com/88250/latke,2026-08-14T17:01:06.101623+00:00,ok,NaN,NaN,True,2021-02-08T21:25:30Z,https://github.com/88250/latke/pull/37,37.0,dependabot,⬆️ Bump netty.version from 4.1.49.Final to 4.1...,app/dependabot,repo:88250/latke is:pr author:app/dependabot c...,False,NaN,NaN,NaN,NaN,NaN



Top error types:
error_type
HTTPError    443
Name: count, dtype: int64


,repo,repo_url,mined_at_utc,mining_status,error_type,error_message
0,42BV/jarb,https://github.com/42BV/jarb,2026-08-14T16:55:34.850082+00:00,error,HTTPError,HTTP Error 401: Unauthorized
1,52North/arctic-sea,https://github.com/52North/arctic-sea,2026-08-14T16:55:35.068779+00:00,error,HTTPError,HTTP Error 401: Unauthorized
2,52North/series-hibernate,https://github.com/52North/series-hibernate,2026-08-14T16:55:35.384423+00:00,error,HTTPError,HTTP Error 401: Unauthorized
3,6tail/nlf2-maven,https://github.com/6tail/nlf2-maven,2026-08-14T16:55:35.636576+00:00,error,HTTPError,HTTP Error 401: Unauthorized
4,88250/latke,https://github.com/88250/latke,2026-08-14T16:55:35.826828+00:00,error,HTTPError,HTTP Error 401: Unauthorized
